# Basketball Analytics — Google Colab

Corre el pipeline completo en GPU de Colab, usando videos subidos a Google Drive.

**Antes de empezar:**
1. Menú **Runtime → Change runtime type → GPU (T4)**
2. Subí los videos a Drive en `Mi unidad/basketball_analytics/videos/`
3. Ejecutar las celdas de arriba a abajo

**Estructura del repo:**
```
supervision-basketball/
├── src/          # módulos del pipeline (ball_tracker, event_engine, etc.)
├── scripts/      # run_pipeline.py, benchmark.py
├── notebooks/    # este archivo
├── data/         # ground_truth.json, videos_url.txt
└── outputs/      # stats JSON, tracking CSV, video anotado
```

## 1. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem:.1f} GB')
else:
    print('⚠️  Sin GPU — cambiá el runtime: Runtime → Change runtime type → GPU')

## 2. Montar Google Drive

Los videos deben estar en Drive. Al montar, Colab puede leerlos directamente.

**Cómo subir los videos:**
- Opción A (recomendada): en el explorador web de Drive, creá `basketball_analytics/videos/` y arrastá los `.mp4`.
- Opción B: desde tu PC con `rclone` o la app de Drive desktop.

Los videos quedan en `/content/drive/MyDrive/basketball_analytics/videos/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/basketball_analytics'
VIDEOS_DIR = f'{DRIVE_ROOT}/videos'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'

os.makedirs(VIDEOS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Videos en  : {VIDEOS_DIR}')
print(f'Outputs en : {OUTPUT_DIR}')
print()
print('Videos disponibles:')
videos = sorted(f for f in os.listdir(VIDEOS_DIR) if f.endswith('.mp4'))
if videos:
    for v in videos:
        mb = os.path.getsize(f'{VIDEOS_DIR}/{v}') / 1e6
        print(f'  {v}  ({mb:.0f} MB)')
else:
    print('  ⚠️  No se encontraron .mp4 — subí los videos a Drive primero.')
    print(f'      Ruta esperada: {VIDEOS_DIR}/')

## 3. Clonar repo e instalar dependencias

> **Nota:** se usa `opencv-python-headless` — en Colab no hay display.

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/sebastianjlopez/pocs.git'
CLONE_DIR = '/content/pocs'
REPO_DIR  = f'{CLONE_DIR}/supervision-basketball'
SRC_DIR   = f'{REPO_DIR}/src'

if not os.path.exists(CLONE_DIR):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}
else:
    print('Repo ya clonado, actualizando...')
    !cd {CLONE_DIR} && git pull

os.chdir(REPO_DIR)

# Verificar que src/ existe
assert os.path.exists(SRC_DIR), f'src/ no encontrado en {SRC_DIR} — verificá el git pull'
assert os.path.exists(f'{SRC_DIR}/ball_tracker.py'), 'ball_tracker.py no encontrado en src/'

# Agregar src/ al path de Python
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Mostrar commits recientes para confirmar versión del código
print('\nÚltimos commits:')
!cd {CLONE_DIR} && git log --oneline -3
print(f'\nsrc/ OK: {os.listdir(SRC_DIR)}')

In [ ]:
# Instalar dependencias (opencv-python-headless: Colab no tiene display)
!pip install -q \
    supervision>=0.25.0 \
    ultralytics>=8.3.0 \
    opencv-python-headless>=4.8.0 \
    numpy>=1.24.0 \
    scipy>=1.11.0 \
    av

import supervision, ultralytics, cv2, av
print('supervision:', supervision.__version__)
print('ultralytics:', ultralytics.__version__)
print('opencv:     ', cv2.__version__)
print('PyAV:       ', av.__version__)

## 4. Configuración del pipeline

Editá estas variables antes de correr.

In [ ]:
import os
from pathlib import Path

# ── Video a procesar ─────────────────────────────────────────────────────────
VIDEO_NAME   = 'video_1_9F5q5jCbrMI.mp4'
SOURCE_VIDEO = f'{VIDEOS_DIR}/{VIDEO_NAME}'

# ── Modelo YOLO ──────────────────────────────────────────────────────────────
# 'yolo11n.pt' (rápido) → 'yolo11s.pt' → 'yolo11m.pt' → 'yolo11x.pt' (preciso)
# Pesos fine-tuneados en Drive:
#   WEIGHTS = f'{DRIVE_ROOT}/basketball_best.pt'
WEIGHTS = 'yolo11n.pt'

# ── Clases del modelo ────────────────────────────────────────────────────────
# Modelo COCO genérico:         PLAYER_CLASS=0 (person), BALL_CLASS=32 (sports ball)
# Modelo fine-tuneado Roboflow: PLAYER_CLASS=1, BALL_CLASS=0
PLAYER_CLASS = 0
BALL_CLASS   = 32

# ── Límite de frames ─────────────────────────────────────────────────────────
# None = video completo. 1800 ≈ 60s a 30fps.
MAX_FRAMES = 1800

# ── Output ───────────────────────────────────────────────────────────────────
SAVE_VIDEO   = True
stem         = Path(SOURCE_VIDEO).stem
OUTPUT_VIDEO = f'{OUTPUT_DIR}/{stem}_annotated.mp4'

# Validar
if not os.path.exists(SOURCE_VIDEO):
    print(f'⚠️  No encontrado: {SOURCE_VIDEO}')
    print(f'   Disponibles en {VIDEOS_DIR}:')
    for f in sorted(os.listdir(VIDEOS_DIR)):
        print(f'     {f}')
else:
    mb = os.path.getsize(SOURCE_VIDEO) / 1e6
    print(f'Video fuente  : {SOURCE_VIDEO}  ({mb:.0f} MB)')
    print(f'Modelo        : {WEIGHTS}')
    print(f'Clases        : jugador={PLAYER_CLASS}, pelota={BALL_CLASS}')
    print(f'Max frames    : {MAX_FRAMES if MAX_FRAMES else "todos"}')
    print(f'Video output  : {OUTPUT_VIDEO if SAVE_VIDEO else "no"}')

## 5. Correr el pipeline

Corre en modo `--headless` (sin ventanas OpenCV). Progreso cada 100 frames.

- GPU T4 con `yolo11n`: ~15–25 fps de procesamiento
- CPU: ~2–5 fps

In [ ]:
import os
os.chdir(REPO_DIR)

cmd_parts = [
    f'PYTHONPATH={SRC_DIR}',       # garantiza que src/ sea encontrado
    'python scripts/run_pipeline.py',
    f'--source "{SOURCE_VIDEO}"',
    f'--weights {WEIGHTS}',
    f'--player-class {PLAYER_CLASS}',
    f'--ball-class {BALL_CLASS}',
    '--headless',
]
if SAVE_VIDEO:
    cmd_parts.append(f'--output "{OUTPUT_VIDEO}"')
if MAX_FRAMES:
    cmd_parts.append(f'--max-frames {MAX_FRAMES}')

cmd = ' '.join(cmd_parts)
print('Corriendo:', cmd)
print('-' * 60)
!{cmd}

## 6. Ver estadísticas

In [ ]:
import json, os, shutil
from pathlib import Path

# El pipeline guarda el JSON en el cwd (REPO_DIR)
local_stats = f'{REPO_DIR}/{stem}_stats.json'
local_csv   = f'{REPO_DIR}/{stem}_tracking.csv'

# Copiar a Drive para persistirlos
for src, dst in [
    (local_stats, f'{OUTPUT_DIR}/{stem}_stats.json'),
    (local_csv,   f'{OUTPUT_DIR}/{stem}_tracking.csv'),
]:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copiado a Drive: {dst}')
    else:
        print(f'No encontrado (pipeline no corrió?): {src}')

# Mostrar stats (solo del run local, no de Drive)
if os.path.exists(local_stats):
    with open(local_stats) as f:
        stats = json.load(f)

    print(f'\n=== Estadísticas: {stem} ===')

    teams = stats.get('teams', {})
    print('\n--- Por equipo ---')
    for team, data in teams.items():
        print(f'  Equipo {team}:')
        for k, v in data.items():
            print(f'    {k:20s}: {v}')

    players = stats.get('players', {})
    if players:
        print('\n--- Top jugadores (por tiros) ---')
        for pid, pd in sorted(players.items(),
                              key=lambda x: x[1].get('shots', 0), reverse=True)[:10]:
            print(f"  #{pid} [Eq.{pd.get('team','?')}]  "
                  f"tiros={pd.get('shots',0)}  "
                  f"canastas={pd.get('baskets',0)}  "
                  f"posesiones={pd.get('possessions',0)}")
else:
    print(f'⚠️  {local_stats} no existe — re-corrí la celda 5 primero.')

## 7. Benchmark vs box score oficial

In [ ]:
import os
os.chdir(REPO_DIR)

local_stats  = f'{REPO_DIR}/{stem}_stats.json'
ground_truth = f'{REPO_DIR}/data/ground_truth.json'

if os.path.exists(local_stats) and os.path.exists(ground_truth):
    !PYTHONPATH={SRC_DIR} python scripts/benchmark.py --stats "{local_stats}" --ground-truth "{ground_truth}"
else:
    missing = [p for p in [local_stats, ground_truth] if not os.path.exists(p)]
    print('Archivos no encontrados:', missing)

## 8. Ver video anotado en el notebook

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

video_path = OUTPUT_VIDEO if SAVE_VIDEO else None

if video_path and os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / 1e6
    print(f'Video: {video_path} ({size_mb:.1f} MB)')
    if size_mb < 50:
        with open(video_path, 'rb') as f:
            b64 = b64encode(f.read()).decode()
        display(HTML(f'<video width="800" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    else:
        print(f'Video muy grande para embeber ({size_mb:.0f} MB).')
        print(f'Abrilo desde Drive: {video_path}')
else:
    print('No se generó video anotado.')

## 9. (Opcional) Batch: todos los videos

Corre el pipeline en todos los `.mp4` de `VIDEOS_DIR`.
Tiempo estimado con `yolo11n` en T4: ~3–5 min por video de 90s.

In [ ]:
import os, shutil
os.chdir(REPO_DIR)

videos = sorted(f for f in os.listdir(VIDEOS_DIR) if f.endswith('.mp4'))
print(f'Videos a procesar: {len(videos)}')

for video_name in videos:
    src  = f'{VIDEOS_DIR}/{video_name}'
    name = video_name.replace('.mp4', '')
    out  = f'{OUTPUT_DIR}/{name}_annotated.mp4'

    print(f'\n{"="*60}')
    print(f'Procesando: {video_name}')
    print(f'{"="*60}')

    !PYTHONPATH={SRC_DIR} python scripts/run_pipeline.py \
        --source "{src}" \
        --output "{out}" \
        --weights {WEIGHTS} \
        --player-class {PLAYER_CLASS} \
        --ball-class {BALL_CLASS} \
        --headless

    for ext in ['_stats.json', '_tracking.csv']:
        local = f'{REPO_DIR}/{name}{ext}'
        if os.path.exists(local):
            shutil.copy2(local, f'{OUTPUT_DIR}/{name}{ext}')

print('\nBatch completo. Outputs en:', OUTPUT_DIR)

## 10. (Opcional) Pesos fine-tuneados

Si ya entrenaste con `notebooks/train_yolo.ipynb` y guardaste `best.pt` en Drive.

In [ ]:
import os

FINE_TUNED_WEIGHTS = f'{DRIVE_ROOT}/basketball_best.pt'

if not os.path.exists(FINE_TUNED_WEIGHTS):
    print(f'No se encontró: {FINE_TUNED_WEIGHTS}')
    print('Corré notebooks/train_yolo.ipynb primero y guardá best.pt en Drive.')
else:
    mb = os.path.getsize(FINE_TUNED_WEIGHTS) / 1e6
    print(f'Pesos encontrados: {FINE_TUNED_WEIGHTS} ({mb:.0f} MB)')

    FT_PLAYER_CLASS = 1
    FT_BALL_CLASS   = 0

    os.chdir(REPO_DIR)
    out_ft = f'{OUTPUT_DIR}/{stem}_finetuned.mp4'

    !PYTHONPATH={SRC_DIR} python scripts/run_pipeline.py \
        --source "{SOURCE_VIDEO}" \
        --output "{out_ft}" \
        --weights "{FINE_TUNED_WEIGHTS}" \
        --player-class {FT_PLAYER_CLASS} \
        --ball-class {FT_BALL_CLASS} \
        --headless

    local_stats = f'{REPO_DIR}/{stem}_stats.json'
    !PYTHONPATH={SRC_DIR} python scripts/benchmark.py \
        --stats "{local_stats}" \
        --ground-truth "{REPO_DIR}/data/ground_truth.json"

## 11. Reporte para Claude

Genera un bloque de texto con todo el contexto — copialo y pegalo en el chat.

In [ ]:
import json, os, textwrap
from pathlib import Path
from datetime import datetime

lines = []

def h(title):
    lines.append(f"\n{'='*60}")
    lines.append(f"  {title}")
    lines.append(f"{'='*60}")

def add(label, value):
    lines.append(f"  {label:<28}: {value}")

h("ENTORNO")
try:
    import torch
    if torch.cuda.is_available():
        add("GPU", torch.cuda.get_device_name(0))
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        add("VRAM total", f"{mem:.1f} GB")
    else:
        add("GPU", "NO DISPONIBLE (CPU mode)")
except Exception as e:
    add("GPU", f"error: {e}")

add("Python", __import__('sys').version.split()[0])
try:
    import ultralytics, supervision, cv2
    add("ultralytics", ultralytics.__version__)
    add("supervision", supervision.__version__)
    add("opencv", cv2.__version__)
except Exception as e:
    add("deps", f"error: {e}")

h("CONFIGURACIÓN")
try:
    add("Video fuente", VIDEO_NAME)
    add("Modelo YOLO", WEIGHTS)
    add("Clases (player/ball)", f"{PLAYER_CLASS} / {BALL_CLASS}")
    add("Max frames", MAX_FRAMES if MAX_FRAMES else "todos")
    add("Output video", OUTPUT_VIDEO if SAVE_VIDEO else "no")
except NameError as e:
    add("config", f"variable no definida: {e}")

h("ESTADÍSTICAS DEL PIPELINE")
try:
    # Solo leer stats locales (del run actual), NO de Drive
    stats_path = f'{REPO_DIR}/{stem}_stats.json'
    if not os.path.exists(stats_path):
        lines.append(f"  ⚠️  No hay stats locales. Re-corrí la celda 5 primero.")
        lines.append(f"      (No se leen stats viejos de Drive para evitar confusión)")
    else:
        with open(stats_path) as f:
            stats = json.load(f)

        teams = stats.get('teams', {})
        if teams:
            lines.append("\n  --- Equipos ---")
            for team, data in teams.items():
                lines.append(f"  Equipo {team}:")
                for k, v in data.items():
                    add(f"    {k}", v)

        players = stats.get('players', {})
        if players:
            lines.append("\n  --- Top jugadores (por tiros) ---")
            for pid, pd in sorted(players.items(),
                                  key=lambda x: x[1].get('shots', 0), reverse=True)[:10]:
                lines.append(f"  #{pid} [Eq.{pd.get('team','?')}]  "
                             f"tiros={pd.get('shots',0)}  "
                             f"canastas={pd.get('baskets',0)}  "
                             f"posesiones={pd.get('possessions',0)}")

        lines.append("\n  --- JSON completo ---")
        lines.append(textwrap.indent(json.dumps(stats, indent=2, ensure_ascii=False), "  "))
except Exception as e:
    lines.append(f"  ERROR leyendo stats: {e}")

lines.append(f"\n{'='*60}")
lines.append(f"  Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"{'='*60}\n")

report = "\n".join(lines)
print(report)

report_path = f'{OUTPUT_DIR}/{stem}_reporte_claude.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"\nReporte guardado en Drive: {report_path}")
print("→ Copiá el texto de arriba y pegalo en el chat con Claude.")